## 线程隔离的持久化层

In [2]:
from langchain_deepseek import ChatDeepSeek
import os
from langgraph.graph import StateGraph, MessagesState, START, END
from dotenv import load_dotenv

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    temperature=0,
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url=os.environ.get("DEEPSEEK_API_BASE"),
)

def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")
graph = builder.compile()

没有激活持久化层，无法实现多轮对话

In [4]:
input_message = {"role": "user", "content": "hi!我是starry"}
for chunk in graph.stream({"messages":[input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

input_message = {"role": "user", "content": "我叫什么名字"}
for chunk in graph.stream({"messages":[input_message]}, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

hi!我是starry
================================== Ai Message ==================================

嗨，Starry！✨ 很高兴认识你～ 我是你的智能助手，无论是学习、工作还是日常问题，都可以随时找我聊聊。今天有什么想探索的吗？比如星空、故事，或者某个让你好奇的知识点？ 🌟
================================ Human Message =================================

我叫什么名字
================================== Ai Message ==================================

你还没有告诉我你的名字呢！😊 你可以告诉我你叫什么，这样我们就能更好地聊天啦～


激活持久化层

In [5]:
from langgraph.checkpoint.memory import MemorySaver
#使用MemorySaver保存中间状态
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [7]:
config = {"configurable": {"thread_id": "1"}}
input_message = {"role": "user", "content": "hi!我是starry"}
for chunk in graph.stream({"messages":[input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

input_message = {"role": "user", "content": "我叫什么名字"}
for chunk in graph.stream({"messages":[input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

hi!我是starry
================================== Ai Message ==================================

哈哈，我记得你，Starry！🌟 刚才已经打过招呼啦，不过再听一遍你的名字还是很开心～  

这次想从哪里开始呢？是聊聊宇宙的浪漫，还是分享你最近的小灵感？或者，想让我用“星空”为你写首诗、画幅画（用文字描述的那种🌌）？  

等你来点亮话题～✨
================================ Human Message =================================

我叫什么名字
================================== Ai Message ==================================

你的名字是 **Starry** 呀～✨  
（我记性可好啦，你第一次打招呼就说“hi!我是starry”，第二次还特意又告诉我一遍，想忘都难呢😉）  

需要我做点什么吗？比如用你的名字编个小故事？🌌


注意thread_id的输入

In [ ]:
input_message = {"role": "user", "content": "我叫什么名字"}
for chunk in graph.stream(
    {"messages":[input_message]}, 
    {"configurable": {"thread_id": 2}}, 
    stream_mode="values"
):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

我叫什么名字
================================== Ai Message ==================================

你的名字是 **Starry** 呀～🌌  
（放心，我可不会忘！毕竟你一开始就郑重介绍了两次呢✨）  

想再确认一遍的话——要不要我拿你的名字写首诗？😉


## 跨线程共享持久化数据
---
- userid
设置内存记忆

In [1]:
from langgraph.store.memory import InMemoryStore
from langchain_openai import OpenAIEmbeddings
import os

import operator
from typing import Annotated,Sequence
from typing_extensions import TypedDict

from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_core.runnables.config import RunnableConfig
from langgraph.graph import StateGraph,START,END
from IPython.display import Image,display
from dotenv import load_dotenv
load_dotenv()

from langgraph.store.base import BaseStore
import uuid

from langgraph.graph import MessagesState
from langgraph.checkpoint.memory import MemorySaver

from langgraph.runtime import Runtime

# 使用OpenAI的封装,但是运行国产嵌入模型
# 使用内存存储来保存向量化后记忆数据
in_memory_store = InMemoryStore(
    index={
        "embed": OpenAIEmbeddings(
            model="Pro/BAAI/bge-m3",
            api_key=os.environ.get("DEEPSEEK_API_KEY"),
            base_url=os.environ.get("DEEPSEEK_API_BASE") + "/v1",
        ),
        "dims": 1024,
    }
)

# 注意: 我们将 Store 参数传递给节点 ---
# 这是我们编译图时使用的 Store
def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    # 从配置中检索用户信息
    user_id = config["configurable"]["user_id"]
    # 从存储中检索记忆
    namespace = ("memories", user_id)
    memories = store.search(namespace, query=str(state["messages"][-1].content))
    info = "\n".join([d.value["data"] for d in memories])
    system_msg = f"你是一个正在与用户交谈的小助手。用户信息: {info}"

    # 如果用户要求模型记住信息，则存储新的记忆
    last_message = state["messages"][-1]
    if "记住" in last_message.content.lower() or "remember" in last_message.content.lower():
        # 硬编码一个记忆
        memory = "用户名字是starrieu"
        store.put(namespace, str(uuid.uuid4()), {"data": memory})

    response = model.invoke(
        [{"role": "system", "content": system_msg}] + state["messages"]
    )
    return {"messages": response}

builder = StateGraph(MessagesState)
builder.add_node("call_model",call_model)
builder.add_edge(START,"call_model")

#注意：我们在编译图时传递了store对象
graph = builder.compile(checkpointer=MemorySaver(), store=in_memory_store)

config = {"configurable": {"thread_id": "1", "user_id": "1"}}

for chunk in graph.stream(
    {"messages": [{"role":"user","content":"hello"}]},
    config,
    stream_mode="values"
):
    print(chunk)


ImportError: cannot import name 'ContextOverflowError' from 'langchain_core.exceptions' (c:\myVScodeProjects\LangGraph_learning\langgraph-env\Lib\site-packages\langchain_core\exceptions.py)

注意线程ID和用户ID

In [14]:
import importlib.metadata

print(importlib.metadata.version("langgraph"))
print(importlib.metadata.version("langgraph-checkpoint"))

1.2.4
4.1.1


In [12]:
config = {"configurable": {"thread_id": "1", "user_id": "1"}}
input_message = {"role": "user", "content": "请记住我的名字叫tomiezhang!"}

for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

请记住我的名字叫tomiezhang!
{}
================================ Human Message =================================

请记住我的名字叫tomiezhang!


## 短期记忆的实现
---
- 基于最简单的ReAct智能体

In [1]:
from typing import Literal

from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode

# 注意: 使用内存存储来存储记忆
memory = MemorySaver()

@tool
def search(query: str):
    """调用此函数可以浏览网络。"""
    # 模拟一个网络搜索返回
    return "北京天气晴朗 大约22度 湿度30%"

tools = [search]
tool_node = ToolNode(tools)
model = ChatDeepSeek(
    model = "deepseek-v4-flash",
    temperature = 0,
    api_key = os.environ.get("DEEPSEEK_API_KEY"),
    base_url = os.environ.get("DEEPSEEK_API_BASE"),
)
bound_model = model.bind_tools(tools)

def should_continue(state: MessagesState):
    """返回下一个要执行的节点。"""
    last_message = state["messages"][-1]
    # 如果没有函数调用, 则结束
    if not last_message.tool_calls:
        return END
    # 否则如果有, 我们继续
    return "action"


# 定义调用模型的函数
def call_model(state: MessagesState):
    response = bound_model.invoke(state["messages"])
    # 我们返回一个列表, 因为这会被添加到现有列表中
    return {"messages": response}

# 定义一个图
workflow = StateGraph(MessagesState)

# 定义我们将在其间循环的两个节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 将入口点设置为 `agent`
# 这意味着第一个被调用的节点是 `agent` 节点
workflow.add_edge(START, "agent")

# 现在我们添加一个条件边
workflow.add_conditional_edges(
    # 首先, 我们定义起始节点。我们使用 `agent`。
    # 这意味着这些是在 `agent` 节点被调用后采取的边。
    "agent",
    # 接下来, 我们传入将确定下一个调用哪个节点的函数。
    should_continue,
    # 接下来, 我们传入路径映射 - 这条边可能去往的所有可能节点
    ["action", END]
)

workflow.add_edge("action","agent")

app = workflow.compile(checkpointer=memory)

ImportError: cannot import name 'ToolNode' from 'langgraph.prebuilt' (unknown location)